In [27]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import poisson
import seaborn as sns
from functools import reduce


In [3]:
!ls

feature_engineering.ipynb


In [4]:
customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
returns = pd.read_csv("../data/returns.csv")

In [5]:
customers.head()

,customer_id,signup_date,country,acquisition_channel,newsletter_score,age_bucket
0,C000000,2023-01-22,CH,referral,8,35-44
1,C000001,2022-11-06,DE,paid_social,5,NaN
2,C000002,2022-07-01,NL,paid_search,4,18-24
3,C000003,2022-11-16,AT,organic,8,45-54
4,C000004,2022-11-08,DE,paid_search,7,NaN


In [6]:
orders.head()

,order_id,customer_id,order_date,channel,revenue,items,store_id,campaign_code
0,O00227143,C044241,2024-05-08,online,184.60,2,NaN,NaN
1,O00090916,C017716,2023-10-08,online,315.22,3,NaN,NaN
2,O00114747,C022306,2023-11-02,store,101.22,1,S025,NaN
3,O00088831,C017277,2024-12-13,online,115.44,1,NaN,NaN
4,O00219066,C042678,2024-04-11,online,70.47,1,NaN,NaN


In [7]:
returns.head()

,return_id,order_id,return_date,reason_code,refund_amount
0,R00000000,O00134258,2024-10-30,not_as_described,326.48
1,R00000001,O00219546,2024-06-19,Too_Small,149.91
2,R00000002,O00002328,2024-06-11,changed_mind,67.60
3,R00000003,O00046483,2024-12-12,wrong_size,NaN
4,R00000004,O00172721,2023-10-30,TOO SMALL,NaN


In [ ]:
returns[returns["return_id"]]

In [15]:
import pandas as pd

def as_of_date_generic(data: pd.DataFrame, date_column: str, as_of_date: str) -> pd.DataFrame:
    as_of_date = pd.to_datetime(as_of_date)
    data[date_column] = pd.to_datetime(data[date_column])
    
    return data[data[date_column] <= as_of_date].copy()

returns_filtered = as_of_date_generic(returns, "return_date", "2024-06-30")
orders_filtered = as_of_date_generic(orders, "order_date", "2024-06-30")

In [21]:
def rfm_feature_maker(
        customers: pd.DataFrame,
        orders: pd.DataFrame,
        as_of_date: str) -> pd.DataFrame:
    as_of_date = pd.to_datetime(as_of_date)

    # Aggregate orders
    agg = orders.groupby("customer_id").agg(
        last_order_date=("order_date", "max"),
        frequency=("order_id", "count"),
        monetary=("revenue", "sum")
    ).reset_index()

    # Recency
    agg["recency_days"] = (as_of_date - agg["last_order_date"]).dt.days

    # Merge with customers
    df = customers[["customer_id", "signup_date"]].merge(agg, on="customer_id", how="left")  # we dont loose any data here

    # Fill customers with no orders
    df["frequency"] = df["frequency"].fillna(0)
    df["monetary"] = df["monetary"].fillna(0)
    df["recency_days"] = df["recency_days"].fillna((as_of_date - pd.to_datetime(df["signup_date"])).dt.days)

    return df[["customer_id", "recency_days", "frequency", "monetary"]]

rfm = rfm_feature_maker(customers, orders_filtered, "2024-06-30")
print(rfm.head())

  customer_id  recency_days  frequency  monetary
0     C000000         160.0        4.0    958.82
1     C000001          53.0        2.0    279.35
2     C000002         333.0        1.0     86.77
3     C000003         592.0        0.0      0.00
4     C000004         178.0        1.0    193.62


In [22]:
def order_feature_maker(orders: pd.DataFrame) -> pd.DataFrame:
    agg = orders.groupby("customer_id").agg(
        avg_items_per_order=("items", "mean"), # we can also use median here.
        total_items=("items", "sum"),
        avg_revenue_per_order=("revenue", "mean"),
        total_revenue=("revenue", "sum")
    ).reset_index()

    return agg

order_features = order_feature_maker(orders_filtered)
print(order_features.head())

  customer_id  avg_items_per_order  total_items  avg_revenue_per_order  \
0     C000000                  2.0            8                239.705   
1     C000001                  1.5            3                139.675   
2     C000002                  1.0            1                 86.770   
3     C000004                  3.0            3                193.620   
4     C000006                  1.5            3                129.580   

   total_revenue  
0         958.82  
1         279.35  
2          86.77  
3         193.62  
4         259.16  


In [25]:
def return_feature_maker(returns: pd.DataFrame) -> pd.DataFrame:
    returns = returns.merge(orders[["order_id", "customer_id"]], on="order_id", how="left")
    agg = returns.groupby("customer_id").agg(
        return_count=("return_id", "count"),
        total_return_value=("refund_amount", "sum"),
        avg_return_value=("refund_amount", "mean")
    ).reset_index()

    return agg

return_features = return_feature_maker(returns_filtered)
print(return_features.head())

  customer_id  return_count  total_return_value  avg_return_value
0     C000000             4                0.00               NaN
1     C000006             1                0.00               NaN
2     C000007             1              115.65            115.65
3     C000013             1                0.00              0.00
4     C000014             2                0.00               NaN


In [ ]:




def is_dormant(mu, t_delta, threshold=.8):
    """Dormancy.
    This feature detects which customers are dormant.
    This is only applicable for customers with at least 2 orders.


    Args:
        mu (_type_): averate time between orders for a customer
        t_delta (_type_): time since last order for a customer
        threshold (float, optional): Based on this we say whether a customer 
        is dormant. Defaults to .8.

    Returns:
        _type_: _description_
    """
    pr = poisson.cdf(k=t_delta, mu=mu)
    return pr > threshold

def build_features(
        customers: pd.DataFrame,
        orders: pd.DataFrame,
        returns: pd.DataFrame,
        as_of_date: str
    ) -> pd.DataFrame:
    # 1. Filter point-in-time
    orders_as_of_date = as_of_date_generic(
        data=orders,
        date_column="order_date",
        as_of_date=as_of_date)
    returns_as_of_date = as_of_date_generic(
        data=returns,
        date_column="return_date",
        as_of_date=as_of_date)

    # rfm features
    rfm = rfm_feature_maker(customers, orders_as_of_date, as_of_date)

    # order features
    order_features = order_feature_maker(orders_as_of_date)

    # return features
    return_features = return_feature_maker(returns_as_of_date)
    
    # mergin all the features together
    features = [rfm, order_features, return_features]
    all_features = reduce(lambda x, y: pd.merge(x, y, on = 'customer_id'), features)

    return all_features


def run_pipeline(customers_path: str, orders_path: str, returns_path: str):
    customers = pd.read_csv(customers_path)
    orders = pd.read_csv(orders_path)
    returns = pd.read_csv(returns_path)

    snapshots = ["2024-06-30", "2024-12-31"]

    for dd in snapshots:
        features = build_features(customers, orders, returns, dd)

        output_path = f"../output/features_{dd}.parquet"
        features.to_parquet(output_path, index=False)

        print(f"[INFO] Snapshot {dd}: {len(features)} customers")

[INFO] Snapshot 2024-06-30: 25105 customers
[INFO] Snapshot 2024-12-31: 31004 customers


In [ ]:
run_pipeline("../data/customers.csv", "../data/orders.csv", "../data/returns.csv")